In [1]:
!pip install -q ctranslate2 peft transformers sentencepiece huggingface_hub pyyaml safetensors bitsandbytes "torchao>=0.16.0"


In [2]:
import shutil
import subprocess

import ctranslate2
import peft
import torch
import transformers
import yaml
from huggingface_hub import HfApi, login
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")

HF_USER = "coded-by-49"
BASE_ID = "facebook/nllb-200-distilled-1.3B"
ADAPTER_ID = "coded-by-49/LANGBRIDGE_AI_IGBO_1.0"
ADAPTER_REVISION = "epoch 1.33 adapter first main run"  # epoch 1.33 adapter
BEAM_SIZE = 4   # match your evaluation's num_beams
MAX_LEN = 256


login(token=token)
api = HfApi()

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("ctranslate2", ctranslate2.__version__)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


torch 2.11.0+cpu
transformers 5.20.2
peft 0.21.2
ctranslate2 4.9.0


# Loading and Merging Adapter

In [3]:
# Hard-coded token removed. `token` is read from Kaggle secrets (HF_TOKEN) in the cell above.

c = "facebook/nllb-200-distilled-1.3B"
adapter_model_id = "coded-by-49/LANGBRIDGE_AI_IGBO_1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)


tokenizer = AutoTokenizer.from_pretrained(
    adapter_model_id,
    src_lang="eng_Latn",
    tgt_lang="ibo_Latn",
)

base_model =  AutoModelForSeq2SeqLM.from_pretrained(
    base_model_id,
    dtype=torch.float32,
    device_map = "auto",
)

langbridge_ai_igbo = PeftModel.from_pretrained(
    base_model,
    adapter_model_id,
    revision="main",
)

merged_model = langbridge_ai_igbo.merge_and_unload()

merged_model.save_pretrained("merged_eng_igbo_v1", safe_serialization=True)
tokenizer.save_pretrained("merged_eng_igbo_v1")

del base_model, merged_model, langbridge_ai_igbo

tokenizer_config.json:   0%|          | 0.00/3.71k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.48GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (4 outputs left out)

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  189MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
def convert_to_ct2(model_path, out_dir):
    # shutil.rmtree(out_dir, ignore_errors=True)

    subprocess.run(
        ["ct2-transformers-converter",
         "--model", model_path,
         "--output_dir", out_dir,
         "--quantization", "int8"],
        check=True,
    )

    tokenizer.save_pretrained(out_dir)


convert_to_ct2("merged_eng_igbo_v1", "ct2_eng_ibo_int8") # from this point I am making use of standard naming igbo-ibo

convert_to_ct2(merged_eng_igbo_v1, "ct2_nllb_base_1.3b_int8")

httpx.HTTPStatusError: Client error '404 Not Found' for url 'https://huggingface.co/merged-eng-ibo/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py", line 537, in cached_files
    hf_hub_download(
    ~~~~~~~~~~~~~~~^
        path_or_repo_id,
        ^^^^^^^^^^^^^^^^
    ...<10 lines>...
        tqdm_class=tqdm_class,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py", line 89, in _inner_fn
    return fn(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_http.py", line 266, in _inner
    return fn(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py", line 1047, in hf_hub_download
    re

CalledProcessError: Command '['ct2-transformers-converter', '--model', 'merged-eng-ibo', '--output_dir', 'ct2-eng-ibo-int8', '--quantization', 'int8']' returned non-zero exit status 1.